# Notebook 1: Extract, Transform and Load (Parts A & B)

In this notebook we take the raw sales file from the branches (`sales.csv`), clean it, and save the clean version (`cleaned_sales.csv`) into a folder that acts like cloud storage.

**How the work is split:**
- **`src/etl_pipeline.py`** does all the cleaning. Every cleaning rule is written once, in the script.
- **This notebook** explores the raw data to find the problems, then calls the script's functions and shows what changed and why. It doesn't repeat any of the cleaning code.

| Step | What we do | Where the code is |
|---|---|---|
| Extract | Read the raw CSV file | `etl.extract()` |
| Explore | Look for problems in the data before fixing anything | this notebook |
| Transform | Remove duplicates, deal with missing values, fix dates, drop negative quantities, add `total_amount` | `etl.transform()` |
| Load | Save `cleaned_sales.csv` and copy it into `cloud_storage_bucket/` | `etl.load()` |

In [1]:
# Setup: load the libraries and our ETL script
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import etl_pipeline as etl   # our main ETL script

pd.set_option("display.width", 120)
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "sales.csv"

## 1. Extract
We read every column as **text** at first. If we let pandas guess the types straight away, some problems get hidden. For example, a blank quantity quietly turns into a decimal number, and dates in different formats are left mixed up.

In [2]:
# Read the raw file and show the first few rows
raw = etl.extract(RAW_FILE)
print(raw.shape)
raw.head()

2026-10-15 15:57:47,887 | INFO    | EXTRACT  | 105 rows read from sales.csv


(105, 7)


,transaction_id,transaction_date,branch,product,category,quantity,unit_price
0,T001,2026-01-03,Kampala,Laptop,Electronics,2,2500000
1,T002,03/01/2026,Mbarara,Printer,Electronics,3,650000
2,T003,2026/01/04,Jinja,Office Chair,Furniture,5,350000
3,T004,05-01-2026,Kampala,Desk,Furniture,2,480000
4,T005,2026-01-06,Mbarara,Keyboard,Accessories,10,75000


## 2. Explore the raw data
Before cleaning anything, we look for each problem mentioned in the assignment. This is exploration only: nothing is changed here. It shows *why* the script needs each cleaning rule.

In [3]:
# 2a. How many values are missing in each column?
raw.isna().sum().to_frame("missing_values").T

,transaction_id,transaction_date,branch,product,category,quantity,unit_price
missing_values,0,0,0,0,0,0,0


**Why remove the row instead of filling in the gap?** The only missing value is the `quantity` for T028. If we filled it with an average, we would be making up a laptop sale worth about UGX 2.5–5 million, and every revenue total would be wrong.

The rule we follow:
- If a **number** we need for revenue is missing (quantity or price), we remove the row.
- If only a **description** is missing (branch, product or category), we keep the row and write `"Unknown"`, because the sale itself is still real.

The removed row isn't lost. It's saved in a quarantine file so the branch can correct it.

In [4]:
# 2b. Are there duplicates? (rows that appear twice, or the same transaction ID used twice)
print("Exact duplicate rows   :", raw.duplicated().sum())
print("Repeated transaction_id:", raw["transaction_id"].duplicated().sum())
raw[raw["transaction_id"].duplicated(keep=False)]

Exact duplicate rows   : 1
Repeated transaction_id: 1


,transaction_id,transaction_date,branch,product,category,quantity,unit_price
9,T010,2026-01-10,Kampala,Desk,Furniture,4,480000
100,T010,2026-01-10,Kampala,Desk,Furniture,4,480000


In [5]:
# 2c. Which date formats are used? (every digit is shown as 9 so we can see the pattern)
patterns = raw["transaction_date"].str.replace(r"\d", "9", regex=True)
patterns.value_counts().to_frame("rows")

,rows
transaction_date,
9999-99-99,101
99/99/9999,2
9999/99/99,1
99-99-9999,1


In [6]:
# Show the rows whose date is NOT already in the YYYY-MM-DD format
raw[patterns != "9999-99-99"][["transaction_id", "transaction_date"]]

,transaction_id,transaction_date
1,T002,03/01/2026
2,T003,2026/01/04
3,T004,05-01-2026
7,T008,08/01/2026


**Which date format is it?** A date like `03/01/2026` could mean 3 January or 1 March. The transaction IDs go up roughly one per day (T001 is 3 January, T003 is 4 January), so T002 must be **3 January**. That tells us these dates are written **day first**. So instead of letting the computer guess, we give it the exact list of formats to use.

In [7]:
# 2d. Are there any negative or zero quantities?
qty = pd.to_numeric(raw["quantity"], errors="coerce")
raw[(qty <= 0)]

,transaction_id,transaction_date,branch,product,category,quantity,unit_price
101,T101,2026-04-11,Mbarara,Desk,Furniture,-2,480000


## 3. Transform: run the script's cleaning function
All the cleaning happens in one call to `etl.transform()`. It returns three things:
- `clean`: the cleaned data
- `rejected`: the rows that were removed, each with the reason
- `report`: a count of what was fixed

The cleaning steps inside the function, in order:
1. Remove duplicates (keep the first copy of each transaction)
2. Remove rows missing a number we need for revenue (quantity or price); write `"Unknown"` for a missing branch, product or category
3. Turn `transaction_date` into a real date, using the day-first formats we found above
4. Remove sales with a negative (or zero) quantity
5. Add `total_amount = quantity x unit_price`
6. Final checks: stop with an error if any duplicates, gaps or bad quantities are left

In [9]:
# Run the cleaning function from the script (this is the only place cleaning happens)
clean, rejected, report = etl.transform(raw)

# A count of what was fixed
pd.Series(report).to_frame("count")

2026-10-15 15:58:12,084 | INFO    | TRANSFORM| rejected 1 row(s): exact duplicate record -> T010
2026-10-15 15:58:12,091 | INFO    | TRANSFORM| rejected 1 row(s): missing critical value (id/date/quantity/price) -> T028
2026-10-15 15:58:12,114 | INFO    | TRANSFORM| rejected 1 row(s): non-positive quantity -> T101
2026-10-15 15:58:12,127 | INFO    | TRANSFORM| 102 clean rows, 3 rejected


,count
rows_in,105
duplicates_removed,1
missing_critical_removed,1
descriptive_values_filled,0
dates_reformatted,45
bad_dates_removed,0
invalid_quantity_removed,1
invalid_price_removed,0
rows_out,102
rows_rejected,3


## 4. Check the results
Now we look at what the script did to each problem we found in section 2.

In [10]:
# Problems 1, 2 and 4: the rows that were removed, and the reason for each
rejected

,transaction_id,transaction_date,branch,product,category,quantity,unit_price,rejection_reason
0,T010,2026-01-10,Kampala,Desk,Furniture,4,480000,exact duplicate record
1,T028,2026-01-28,Kampala,Laptop,Electronics,,2500000,missing critical value (id/date/quantity/price)
2,T101,2026-04-11,Mbarara,Desk,Furniture,-2,480000,non-positive quantity


In [11]:
# Problem 3: the dates before and after cleaning
ids = ["T002", "T003", "T004", "T008"]
before = raw.set_index("transaction_id").loc[ids, "transaction_date"]
after = clean.set_index("transaction_id").loc[ids, "transaction_date"].dt.strftime("%Y-%m-%d")
pd.DataFrame({"before": before, "after": after})

,before,after
transaction_id,,
T002,03/01/2026,2026-01-03
T003,2026/01/04,2026-04-01
T004,05-01-2026,2026-01-05
T008,08/01/2026,2026-01-08


In [12]:
# Problem 5: the new total_amount column
print("Rows before:", len(raw), "| rows after:", len(clean), "| rows removed:", len(rejected))
clean.head()

Rows before: 105 | rows after: 102 | rows removed: 3


,transaction_id,transaction_date,branch,product,category,quantity,unit_price,total_amount
0,T032,2026-01-02,Mbarara,Keyboard,Accessories,9,75000,675000
1,T002,2026-01-03,Mbarara,Printer,Electronics,3,650000,1950000
2,T060,2026-01-03,Jinja,Keyboard,Accessories,8,75000,600000
3,T091,2026-01-04,Kampala,Laptop,Electronics,2,2500000,5000000
4,T004,2026-01-05,Kampala,Desk,Furniture,2,480000,960000


**Final checks.** The script already stops with an error if any of these fail. We show them here as visible proof for the reader.

In [13]:
# Final checks on the clean data 
checks = {
    "no duplicate transaction_id": clean["transaction_id"].is_unique,
    "no missing values":           bool(clean.notna().all().all()),
    "all dates are datetime":      pd.api.types.is_datetime64_any_dtype(clean["transaction_date"]),
    "all quantities > 0":          bool((clean["quantity"] > 0).all()),
    "total_amount correct":        bool((clean["total_amount"] == clean["quantity"] * clean["unit_price"]).all()),
}
pd.Series(checks).to_frame("passed")

,passed
no duplicate transaction_id,True
no missing values,True
all dates are datetime,True
all quantities > 0,True
total_amount correct,True


## 5. Load
Now we save the clean data with the script's `etl.load()` function. We pass in the `clean` and `rejected` data from section 3, so nothing is cleaned twice. It creates these files:

```
data/processed/cleaned_sales.csv                   <- the cleaned file we submit
cloud_storage_bucket/
├── raw/sales/2026/sales.csv                       <- the original file, as it arrived
├── sales/2026/cleaned_sales.csv                   <- the clean data, grouped by year
├── sales/2026/_manifest.json                      <- details about the file (size, rows, time)
└── quarantine/sales/2026/rejected_sales.csv       <- the rows we removed, and why
```

When you run the script itself (`python src/etl_pipeline.py`), it also writes `_pipeline_run_report.json` with a summary of the run.

In [14]:
# Saving the clean data locally and into the simulated cloud bucket
keys = etl.load(clean, rejected, RAW_FILE, etl.DEFAULT_BUCKET, etl.DEFAULT_PROCESSED)

# Listing the files now in the bucket
for p in sorted((PROJECT_ROOT / "cloud_storage_bucket").rglob("*")):
    if p.is_file():
        print(p.relative_to(PROJECT_ROOT))

2026-10-15 16:00:11,500 | INFO    | LOAD     | wrote data/processed/cleaned_sales.csv
2026-10-15 16:00:11,519 | INFO    | LOAD     | s3-style PUT  cloud_storage_bucket/sales/2026/cleaned_sales.csv  (102 rows)
2026-10-15 16:00:11,524 | INFO    | LOAD     | s3-style PUT  cloud_storage_bucket/quarantine/sales/2026/rejected_sales.csv  (3 rows)


cloud_storage_bucket/.DS_Store
cloud_storage_bucket/quarantine/sales/2026/_manifest.json
cloud_storage_bucket/quarantine/sales/2026/rejected_sales.csv
cloud_storage_bucket/quarantine/sales/2026/rejected_sales_manifest.json
cloud_storage_bucket/raw/sales/2026/sales.csv
cloud_storage_bucket/sales/2026/_manifest.json
cloud_storage_bucket/sales/2026/cleaned_sales.csv
cloud_storage_bucket/sales/2026/cleaned_sales_manifest.json


In [15]:
# Show the details saved about the clean file
import json
print(json.dumps(json.loads((PROJECT_ROOT / "cloud_storage_bucket/sales/2026/_manifest.json").read_text()), indent=2))

{
  "bucket": "cloud_storage_bucket",
  "key": "sales/2026/cleaned_sales.csv",
  "content_type": "text/csv",
  "size_bytes": 6266,
  "row_count": 102,
  "md5_etag": "303b0425bb72348f19a5b8ef93b57a81",
  "uploaded_at_utc": "2026-10-15T09:58:41+00:00"
}


### How this folder acts like cloud storage
* **The bucket:** `cloud_storage_bucket/` plays the part of a cloud storage bucket, like Amazon S3 or Google Cloud Storage.
* **Folder-style names:** cloud storage doesn't really have folders. A file is saved under a name like `sales/2026/cleaned_sales.csv`, and the slashes just make it look like folders. We copy that with real folders.
* **Grouped by year:** putting files under `sales/2026/` means a query about 2026 only has to read the 2026 files, which is faster and cheaper.
* **Three areas:** `raw/` keeps the original file untouched, `sales/` holds the clean data, and `quarantine/` keeps the rejected rows so they can be checked later.
* **File details:** each saved file gets a `_manifest.json` with its size, number of rows, a checksum and the time it was saved, like the details a cloud service records.
* **Safe saving:** each file is first written under a temporary name and then renamed, so nobody ever opens a half-saved file.